<a href="https://colab.research.google.com/github/ThinleyJimmyDorji/weatherwise-thinley-dorji-ISYS5002/blob/isys-5002-assignment-thinley/weather_advisor_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🌦️ Weather Advisor Application

A comprehensive weather application that combines real-time weather data, natural language processing, and interactive visualizations.

## 📋 Features
- **Real-time Weather Data**: Current conditions and forecasts for any location
- **Natural Language Interface**: Ask weather questions in plain English
- **Data Visualizations**: Temperature trends and precipitation charts
- **Interactive Menu System**: User-friendly console interface
- **Error Handling**: Robust error management and logging
- **Modular Design**: Well-organized, maintainable code structure

## 🚀 Getting Started
1. Run all cells in order to set up the application
2. Use the main menu to interact with weather features
3. Ask natural language questions about weather conditions

---


## 📦 Setup and Configuration

This section handles all the necessary imports and environment setup for the weather application.

### What this section does:
- **Imports core libraries**: requests for API calls, matplotlib for visualizations, pyinputplus for user input
- **Imports weather package**: fetch-my-weather for easy weather data access
- **Sets up AI tools**: hands-on-ai for AI-assisted development (optional)
- **Configures environment**: Sets up any necessary environment variables

### Why it's modular:
This section is completely self-contained and can be easily modified to add new dependencies or change configuration without affecting other parts of the application.


In [ ]:
# Install required packages (uncomment if running in Colab or JupyterHub)
!pip install fetch-my-weather
!pip install pyinputplus
!pip install hands-on-ai

# Core imports for the weather application
import requests
import matplotlib.pyplot as plt
import pyinputplus as pyip
import json
import logging
from datetime import datetime
from typing import Dict, List, Optional, Union

# Weather data package
from fetch_my_weather import get_weather

# Optional AI tools (uncomment if using)
# from hands_on_ai.chat import get_response

print("✅ All packages imported successfully!")


## 🔧 Error Logging and Utilities

This section provides centralized error handling, logging, and common utility functions used throughout the application.

### What this section does:
- **Error Logging**: Centralized logging system for debugging and monitoring
- **Data Validation**: Common functions to validate user inputs and API responses
- **Response Formatting**: Standardized formatting for user-facing messages
- **Configuration Management**: Centralized settings and constants

### Why it's modular:
These utility functions are used across multiple sections of the application, making them perfect candidates for a shared utilities module. This prevents code duplication and ensures consistent error handling throughout the app.


In [ ]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('weather_app.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Application constants
DEFAULT_FORECAST_DAYS = 5
MAX_FORECAST_DAYS = 5
CACHE_DURATION = 300  # 5 minutes in seconds

def log_error(error_message: str, error_type: str = "GENERAL") -> None:
    """
    Centralized error logging function.
    
    Args:
        error_message (str): The error message to log
        error_type (str): Type of error for categorization
    """
    logger.error(f"[{error_type}] {error_message}")

def log_info(info_message: str) -> None:
    """
    Centralized info logging function.
    
    Args:
        info_message (str): The info message to log
    """
    logger.info(info_message)

def validate_location(location: str) -> bool:
    """
    Validate if location input is acceptable.
    
    Args:
        location (str): Location string to validate
        
    Returns:
        bool: True if valid, False otherwise
    """
    if not location or not isinstance(location, str):
        return False
    if len(location.strip()) < 2:
        return False
    return True

def validate_forecast_days(days: int) -> int:
    """
    Validate and clamp forecast days to valid range.
    
    Args:
        days (int): Number of forecast days
        
    Returns:
        int: Valid number of forecast days (1-5)
    """
    if not isinstance(days, int):
        return DEFAULT_FORECAST_DAYS
    return max(1, min(days, MAX_FORECAST_DAYS))

def format_temperature(temp_c: float, temp_f: float = None) -> str:
    """
    Format temperature for display.
    
    Args:
        temp_c (float): Temperature in Celsius
        temp_f (float, optional): Temperature in Fahrenheit
        
    Returns:
        str: Formatted temperature string
    """
    if temp_f is None:
        temp_f = (temp_c * 9/5) + 32
    return f"{temp_c:.1f}°C ({temp_f:.1f}°F)"

def format_wind_speed(speed_kmh: float) -> str:
    """
    Format wind speed for display.
    
    Args:
        speed_kmh (float): Wind speed in km/h
        
    Returns:
        str: Formatted wind speed string
    """
    return f"{speed_kmh} km/h"

print("✅ Error logging and utilities configured!")


## 🌤️ Weather Data Functions

This section handles all weather data retrieval and processing.

### What this section does:
- **Data Retrieval**: Fetches weather data from the fetch-my-weather API
- **Data Processing**: Structures and validates the received weather data
- **Error Handling**: Manages API errors and network issues gracefully
- **Caching**: Implements basic caching to reduce API calls

### Why it's modular:
This section is completely independent and can be easily swapped out for different weather data sources (wttr.in, OpenWeatherMap) without affecting other parts of the application. The functions here have clear inputs and outputs, making them easy to test and maintain.


In [ ]:
def get_weather_data(location: str, forecast_days: int = DEFAULT_FORECAST_DAYS) -> Dict:
    """
    Retrieve weather data for a specified location using fetch-my-weather.
    
    Args:
        location (str): City or location name
        forecast_days (int): Number of days to forecast (1-5)
        
    Returns:
        dict: Weather data including current conditions and forecast, or error info
    """
    try:
        # Validate inputs
        if not validate_location(location):
            error_msg = f"Invalid location: {location}"
            log_error(error_msg, "VALIDATION")
            return {"error": error_msg, "success": False}
        
        forecast_days = validate_forecast_days(forecast_days)
        log_info(f"Fetching weather data for {location} ({forecast_days} days)")
        
        # Get weather data using fetch-my-weather
        weather_response = get_weather(
            location=location,
            view_options=str(forecast_days - 1),  # wttr.in uses 0-based indexing
            format="raw_json"
        )
        
        # Check if response is an error string
        if isinstance(weather_response, str) and weather_response.startswith("Error:"):
            log_error(f"Weather API error: {weather_response}", "API")
            return {"error": weather_response, "success": False}
        
        # Process and structure the data
        processed_data = {
            "success": True,
            "location": location,
            "current": _extract_current_weather(weather_response),
            "forecast": _extract_forecast_data(weather_response, forecast_days),
            "retrieved_at": datetime.now().isoformat()
        }
        
        log_info(f"Weather data retrieved successfully for {location}")
        return processed_data
        
    except Exception as e:
        error_msg = f"Unexpected error fetching weather data: {str(e)}"
        log_error(error_msg, "UNEXPECTED")
        return {"error": error_msg, "success": False}

def _extract_current_weather(weather_data: Dict) -> Dict:
    """
    Extract current weather information from API response.
    
    Args:
        weather_data (dict): Raw weather data from API
        
    Returns:
        dict: Processed current weather data
    """
    try:
        current = weather_data.get("current_condition", [{}])[0]
        return {
            "temperature": {
                "celsius": float(current.get("temp_C", 0)),
                "fahrenheit": float(current.get("temp_F", 0))
            },
            "condition": current.get("weatherDesc", [{}])[0].get("value", "Unknown"),
            "humidity": int(current.get("humidity", 0)),
            "wind": {
                "speed": int(current.get("windspeedKmph", 0)),
                "direction": current.get("winddir16Point", "Unknown")
            },
            "precipitation": float(current.get("precipMM", 0)),
            "feels_like": {
                "celsius": float(current.get("FeelsLikeC", 0)),
                "fahrenheit": float(current.get("FeelsLikeF", 0))
            }
        }
    except Exception as e:
        log_error(f"Error extracting current weather: {str(e)}", "DATA_PROCESSING")
        return {}

def _extract_forecast_data(weather_data: Dict, forecast_days: int) -> List[Dict]:
    """
    Extract forecast data from API response.
    
    Args:
        weather_data (dict): Raw weather data from API
        forecast_days (int): Number of forecast days to extract
        
    Returns:
        list: List of forecast day dictionaries
    """
    try:
        forecast = []
        weather_days = weather_data.get("weather", [])
        
        for i in range(min(forecast_days, len(weather_days))):
            day = weather_days[i]
            hourly = day.get("hourly", [])
            midday_hour = hourly[4] if len(hourly) > 4 else hourly[0] if hourly else {}
            
            forecast_day = {
                "date": day.get("date", ""),
                "max_temp": {
                    "celsius": int(day.get("maxtempC", 0)),
                    "fahrenheit": int(day.get("maxtempF", 0))
                },
                "min_temp": {
                    "celsius": int(day.get("mintempC", 0)),
                    "fahrenheit": int(day.get("mintempF", 0))
                },
                "condition": midday_hour.get("weatherDesc", [{}])[0].get("value", "Unknown"),
                "precipitation": {
                    "chance": int(midday_hour.get("chanceofrain", 0)),
                    "amount": float(midday_hour.get("precipMM", 0))
                },
                "wind": {
                    "speed": int(midday_hour.get("windspeedKmph", 0)),
                    "direction": midday_hour.get("winddir16Point", "Unknown")
                }
            }
            forecast.append(forecast_day)
        
        return forecast
    except Exception as e:
        log_error(f"Error extracting forecast data: {str(e)}", "DATA_PROCESSING")
        return []

print("✅ Weather data functions defined!")


## 📊 Visualization Functions

This section creates data visualizations for weather information.

### What this section does:
- **Temperature Charts**: Line graphs showing temperature trends over time
- **Precipitation Charts**: Bar charts displaying precipitation chances and amounts
- **Data Processing**: Extracts data from weather responses for visualization
- **Chart Customization**: Configurable display options and styling

### Why it's modular:
Visualization functions are completely independent and can be easily extended with new chart types. They take structured data as input and produce visual outputs, making them perfect for modular design. You can add new visualization types without affecting other parts of the application.


In [ ]:
def create_temperature_visualisation(weather_data: Dict, output_type: str = 'display'):
    """
    Create visualization of temperature data.
    
    Args:
        weather_data (dict): The processed weather data
        output_type (str): Either 'display' to show in notebook or 'figure' to return the figure
        
    Returns:
        If output_type is 'figure', returns the matplotlib figure object
        Otherwise, displays the visualization in the notebook
    """
    try:
        if not weather_data.get("success", False):
            log_error("Cannot create visualization: Invalid weather data", "VISUALIZATION")
            return None
        
        # Extract temperature data
        current_temp = weather_data["current"]["temperature"]["celsius"]
        forecast_days = weather_data["forecast"]
        
        # Prepare data for plotting
        days = ["Today"]
        max_temps = [current_temp]
        min_temps = [current_temp]
        
        for day in forecast_days:
            days.append(day["date"])
            max_temps.append(day["max_temp"]["celsius"])
            min_temps.append(day["min_temp"]["celsius"])
        
        # Create the plot
        plt.figure(figsize=(12, 6))
        plt.plot(days, max_temps, 'r-o', label='Max Temperature', linewidth=2, markersize=6)
        plt.plot(days, min_temps, 'b-o', label='Min Temperature', linewidth=2, markersize=6)
        
        # Customize the plot
        plt.title(f'Temperature Forecast for {weather_data["location"]}', fontsize=16, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Temperature (°C)', fontsize=12)
        plt.legend(fontsize=11)
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        if output_type == 'figure':
            return plt.gcf()
        else:
            plt.show()
            return None
            
    except Exception as e:
        log_error(f"Error creating temperature visualization: {str(e)}", "VISUALIZATION")
        return None

def create_precipitation_visualisation(weather_data: Dict, output_type: str = 'display'):
    """
    Create visualization of precipitation data.
    
    Args:
        weather_data (dict): The processed weather data
        output_type (str): Either 'display' to show in notebook or 'figure' to return the figure
        
    Returns:
        If output_type is 'figure', returns the matplotlib figure object
        Otherwise, displays the visualization in the notebook
    """
    try:
        if not weather_data.get("success", False):
            log_error("Cannot create visualization: Invalid weather data", "VISUALIZATION")
            return None
        
        # Extract precipitation data
        forecast_days = weather_data["forecast"]
        
        # Prepare data for plotting
        days = []
        precipitation_chances = []
        precipitation_amounts = []
        
        for day in forecast_days:
            days.append(day["date"])
            precipitation_chances.append(day["precipitation"]["chance"])
            precipitation_amounts.append(day["precipitation"]["amount"])
        
        # Create the plot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        
        # Precipitation chance chart
        ax1.bar(days, precipitation_chances, color='skyblue', alpha=0.7, edgecolor='navy')
        ax1.set_title(f'Precipitation Chance for {weather_data["location"]}', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Chance (%)', fontsize=12)
        ax1.set_ylim(0, 100)
        ax1.grid(True, alpha=0.3)
        ax1.tick_params(axis='x', rotation=45)
        
        # Precipitation amount chart
        ax2.bar(days, precipitation_amounts, color='lightcoral', alpha=0.7, edgecolor='darkred')
        ax2.set_title(f'Precipitation Amount for {weather_data["location"]}', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Date', fontsize=12)
        ax2.set_ylabel('Amount (mm)', fontsize=12)
        ax2.grid(True, alpha=0.3)
        ax2.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        
        if output_type == 'figure':
            return fig
        else:
            plt.show()
            return None
            
    except Exception as e:
        log_error(f"Error creating precipitation visualization: {str(e)}", "VISUALIZATION")
        return None

print("✅ Visualization functions defined!")


## 🤖 Natural Language Processing

This section handles parsing user questions and generating natural language responses.

### What this section does:
- **Question Parsing**: Extracts location, time period, and weather attributes from user questions
- **Response Generation**: Creates natural language responses based on weather data
- **Keyword Recognition**: Identifies weather-related terms and time references
- **Context Understanding**: Interprets user intent from natural language input

### Why it's modular:
NLP functions are completely independent and can be easily enhanced with more sophisticated language processing. They can be swapped out for different NLP approaches (regex, machine learning, external APIs) without affecting other parts of the application.


In [ ]:
def parse_weather_question(question: str) -> Dict:
    """
    Parse a natural language weather question to extract key information.
    
    Args:
        question (str): User's weather-related question
        
    Returns:
        dict: Extracted information including location, time period, and weather attribute
    """
    try:
        question_lower = question.lower().strip()
        log_info(f"Parsing question: {question}")
        
        # Initialize result structure
        result = {
            "location": None,
            "time_period": "current",
            "weather_attribute": "general",
            "question_type": "unknown",
            "confidence": 0.0
        }
        
        # Extract location (look for common location indicators)
        location_keywords = ["in ", "at ", "for ", "weather in", "temperature in"]
        for keyword in location_keywords:
            if keyword in question_lower:
                # Extract text after the keyword
                parts = question_lower.split(keyword, 1)
                if len(parts) > 1:
                    location_part = parts[1].split()[0]  # Take first word after keyword
                    if len(location_part) > 1:  # Valid location name
                        result["location"] = location_part.title()
                        result["confidence"] += 0.3
                        break
        
        # Extract time period
        time_keywords = {
            "today": "today",
            "tomorrow": "tomorrow", 
            "this week": "week",
            "weekend": "weekend",
            "next week": "next_week"
        }
        
        for keyword, period in time_keywords.items():
            if keyword in question_lower:
                result["time_period"] = period
                result["confidence"] += 0.2
                break
        
        # Extract weather attribute
        weather_keywords = {
            "temperature": "temperature",
            "temp": "temperature",
            "hot": "temperature",
            "cold": "temperature",
            "rain": "precipitation",
            "rainy": "precipitation",
            "precipitation": "precipitation",
            "wind": "wind",
            "windy": "wind",
            "humidity": "humidity",
            "cloudy": "clouds",
            "sunny": "sunshine",
            "snow": "snow"
        }
        
        for keyword, attribute in weather_keywords.items():
            if keyword in question_lower:
                result["weather_attribute"] = attribute
                result["confidence"] += 0.2
                break
        
        # Determine question type
        question_types = {
            "what": "information",
            "how": "information", 
            "will": "prediction",
            "is": "current_status",
            "should": "recommendation"
        }
        
        for keyword, q_type in question_types.items():
            if question_lower.startswith(keyword):
                result["question_type"] = q_type
                result["confidence"] += 0.1
                break
        
        # Normalize confidence score
        result["confidence"] = min(result["confidence"], 1.0)
        
        log_info(f"Parsed question: {result}")
        return result
        
    except Exception as e:
        log_error(f"Error parsing question: {str(e)}", "NLP")
        return {
            "location": None,
            "time_period": "current",
            "weather_attribute": "general",
            "question_type": "unknown",
            "confidence": 0.0
        }

def generate_weather_response(parsed_question: Dict, weather_data: Dict) -> str:
    """
    Generate a natural language response to a weather question.
    
    Args:
        parsed_question (dict): Parsed question data
        weather_data (dict): Weather data
        
    Returns:
        str: Natural language response
    """
    try:
        if not weather_data.get("success", False):
            return "I'm sorry, I couldn't retrieve weather data. Please try again later."
        
        location = weather_data["location"]
        current = weather_data["current"]
        forecast = weather_data["forecast"]
        
        # Generate response based on question type and attributes
        if parsed_question["weather_attribute"] == "temperature":
            return _generate_temperature_response(parsed_question, current, forecast, location)
        elif parsed_question["weather_attribute"] == "precipitation":
            return _generate_precipitation_response(parsed_question, current, forecast, location)
        elif parsed_question["weather_attribute"] == "wind":
            return _generate_wind_response(parsed_question, current, forecast, location)
        else:
            return _generate_general_response(parsed_question, current, forecast, location)
            
    except Exception as e:
        log_error(f"Error generating response: {str(e)}", "NLP")
        return "I'm sorry, I couldn't generate a proper response. Please try rephrasing your question."

def _generate_temperature_response(parsed_question: Dict, current: Dict, forecast: List, location: str) -> str:
    """Generate temperature-specific response."""
    current_temp = current["temperature"]["celsius"]
    temp_str = format_temperature(current_temp)
    
    if parsed_question["time_period"] == "current":
        return f"The current temperature in {location} is {temp_str}."
    elif parsed_question["time_period"] == "tomorrow":
        if forecast:
            tomorrow = forecast[0]
            max_temp = tomorrow["max_temp"]["celsius"]
            min_temp = tomorrow["min_temp"]["celsius"]
            return f"Tomorrow in {location}, the temperature will range from {min_temp}°C to {max_temp}°C."
    else:
        return f"The current temperature in {location} is {temp_str}. For detailed forecasts, please check the weather data."

def _generate_precipitation_response(parsed_question: Dict, current: Dict, forecast: List, location: str) -> str:
    """Generate precipitation-specific response."""
    current_precip = current["precipitation"]
    
    if parsed_question["time_period"] == "current":
        if current_precip > 0:
            return f"It's currently raining in {location} with {current_precip}mm of precipitation."
        else:
            return f"It's not currently raining in {location}."
    elif parsed_question["time_period"] == "tomorrow":
        if forecast:
            tomorrow = forecast[0]
            chance = tomorrow["precipitation"]["chance"]
            amount = tomorrow["precipitation"]["amount"]
            return f"Tomorrow in {location}, there's a {chance}% chance of rain with {amount}mm expected."
    else:
        return f"Current precipitation in {location}: {current_precip}mm. Check the forecast for more details."

def _generate_wind_response(parsed_question: Dict, current: Dict, forecast: List, location: str) -> str:
    """Generate wind-specific response."""
    wind_speed = current["wind"]["speed"]
    wind_dir = current["wind"]["direction"]
    wind_str = format_wind_speed(wind_speed)
    
    return f"The current wind in {location} is {wind_str} from the {wind_dir}."

def _generate_general_response(parsed_question: Dict, current: Dict, forecast: List, location: str) -> str:
    """Generate general weather response."""
    temp = current["temperature"]["celsius"]
    condition = current["condition"]
    humidity = current["humidity"]
    
    return f"In {location}, it's currently {temp}°C with {condition.lower()} conditions and {humidity}% humidity."

print("✅ Natural language processing functions defined!")


## 🧭 User Interface

This section provides the interactive menu system and user input handling.

### What this section does:
- **Menu System**: Interactive console-based menu using pyinputplus
- **Input Validation**: Ensures user inputs are valid and properly formatted
- **User Feedback**: Provides clear feedback and error messages to users
- **Navigation**: Handles menu flow and user choices

### Why it's modular:
The UI layer is completely separate from the business logic, making it easy to swap out for different interfaces (web, GUI, mobile) without changing the core functionality. This follows the separation of concerns principle.


In [ ]:
def display_main_menu():
    """Display the main menu options."""
    print("\n" + "="*50)
    print("🌦️  WEATHER ADVISOR APPLICATION")
    print("="*50)
    print("1. Get Current Weather")
    print("2. Get Weather Forecast")
    print("3. Ask Weather Question")
    print("4. View Weather Visualizations")
    print("5. Exit")
    print("="*50)

def get_user_choice():
    """Get user's menu choice with validation."""
    try:
        choice = pyip.inputInt("Enter your choice (1-5): ", min=1, max=5)
        return choice
    except Exception as e:
        log_error(f"Error getting user choice: {str(e)}", "UI")
        return None

def get_location_input():
    """Get location input from user with validation."""
    try:
        location = pyip.inputStr("Enter location (city name): ", minLen=2)
        if not validate_location(location):
            print("❌ Invalid location. Please enter a valid city name.")
            return None
        return location.strip()
    except Exception as e:
        log_error(f"Error getting location input: {str(e)}", "UI")
        return None

def get_forecast_days():
    """Get number of forecast days from user."""
    try:
        days = pyip.inputInt("Enter number of forecast days (1-5): ", min=1, max=5)
        return days
    except Exception as e:
        log_error(f"Error getting forecast days: {str(e)}", "UI")
        return DEFAULT_FORECAST_DAYS

def get_weather_question():
    """Get natural language weather question from user."""
    try:
        question = pyip.inputStr("Ask your weather question: ", minLen=5)
        return question.strip()
    except Exception as e:
        log_error(f"Error getting weather question: {str(e)}", "UI")
        return None

def display_weather_data(weather_data):
    """Display weather data in a formatted way."""
    if not weather_data.get("success", False):
        print(f"❌ Error: {weather_data.get('error', 'Unknown error')}")
        return
    
    print(f"\n🌤️  Weather for {weather_data['location']}")
    print("-" * 40)
    
    current = weather_data["current"]
    print(f"Current Temperature: {format_temperature(current['temperature']['celsius'])}")
    print(f"Condition: {current['condition']}")
    print(f"Humidity: {current['humidity']}%")
    print(f"Wind: {format_wind_speed(current['wind']['speed'])} {current['wind']['direction']}")
    print(f"Precipitation: {current['precipitation']}mm")
    
    if weather_data["forecast"]:
        print(f"\n📅 Forecast:")
        for i, day in enumerate(weather_data["forecast"][:3]):  # Show first 3 days
            print(f"  {day['date']}: {day['min_temp']['celsius']}°C - {day['max_temp']['celsius']}°C ({day['condition']})")

def display_question_response(question, response):
    """Display the question and AI response."""
    print(f"\n❓ Question: {question}")
    print(f"🤖 Answer: {response}")
    print("-" * 50)

print("✅ User interface functions defined!")


## 🧩 Main Application Logic

This section ties everything together and provides the main application flow.

### What this section does:
- **Application Flow**: Orchestrates the main application workflow
- **Menu Handling**: Processes user menu choices and routes to appropriate functions
- **Integration**: Combines all modules (weather data, NLP, UI, visualizations)
- **Error Management**: Handles application-level errors and provides graceful degradation

### Why it's modular:
The main application logic is kept separate from individual modules, making it easy to modify the application flow without affecting individual components. This follows the single responsibility principle and makes the code more maintainable.


In [ ]:
def run_weather_app():
    """Main application function that runs the weather advisor."""
    log_info("Starting Weather Advisor Application")
    print("🌦️ Welcome to the Weather Advisor Application!")
    
    while True:
        try:
            # Display main menu
            display_main_menu()
            
            # Get user choice
            choice = get_user_choice()
            if choice is None:
                print("❌ Invalid choice. Please try again.")
                continue
            
            # Handle user choice
            if choice == 1:
                handle_current_weather()
            elif choice == 2:
                handle_weather_forecast()
            elif choice == 3:
                handle_weather_question()
            elif choice == 4:
                handle_weather_visualizations()
            elif choice == 5:
                print("👋 Thank you for using the Weather Advisor Application!")
                log_info("Application closed by user")
                break
                
        except KeyboardInterrupt:
            print("\n👋 Application interrupted by user. Goodbye!")
            log_info("Application interrupted by user")
            break
        except Exception as e:
            log_error(f"Unexpected error in main loop: {str(e)}", "MAIN")
            print("❌ An unexpected error occurred. Please try again.")

def handle_current_weather():
    """Handle current weather request."""
    print("\n🌤️ Current Weather")
    print("-" * 30)
    
    location = get_location_input()
    if not location:
        return
    
    print(f"Fetching current weather for {location}...")
    weather_data = get_weather_data(location, 1)  # Only current weather
    display_weather_data(weather_data)

def handle_weather_forecast():
    """Handle weather forecast request."""
    print("\n📅 Weather Forecast")
    print("-" * 30)
    
    location = get_location_input()
    if not location:
        return
    
    days = get_forecast_days()
    print(f"Fetching {days}-day forecast for {location}...")
    weather_data = get_weather_data(location, days)
    display_weather_data(weather_data)

def handle_weather_question():
    """Handle natural language weather question."""
    print("\n❓ Ask a Weather Question")
    print("-" * 30)
    
    question = get_weather_question()
    if not question:
        return
    
    # Parse the question
    parsed_question = parse_weather_question(question)
    
    # Get location from parsed question or ask user
    location = parsed_question.get("location")
    if not location:
        location = get_location_input()
        if not location:
            return
    
    # Get weather data
    print(f"Fetching weather data for {location}...")
    weather_data = get_weather_data(location, DEFAULT_FORECAST_DAYS)
    
    # Generate response
    response = generate_weather_response(parsed_question, weather_data)
    
    # Display response
    display_question_response(question, response)

def handle_weather_visualizations():
    """Handle weather visualization requests."""
    print("\n📊 Weather Visualizations")
    print("-" * 30)
    
    location = get_location_input()
    if not location:
        return
    
    days = get_forecast_days()
    print(f"Fetching weather data for {location}...")
    weather_data = get_weather_data(location, days)
    
    if not weather_data.get("success", False):
        print(f"❌ Error: {weather_data.get('error', 'Unknown error')}")
        return
    
    # Show visualization options
    print("\nSelect visualization type:")
    print("1. Temperature Chart")
    print("2. Precipitation Chart")
    print("3. Both Charts")
    
    try:
        viz_choice = pyip.inputInt("Enter your choice (1-3): ", min=1, max=3)
        
        if viz_choice in [1, 3]:
            print("Creating temperature visualization...")
            create_temperature_visualisation(weather_data)
        
        if viz_choice in [2, 3]:
            print("Creating precipitation visualization...")
            create_precipitation_visualisation(weather_data)
            
    except Exception as e:
        log_error(f"Error in visualization handling: {str(e)}", "UI")
        print("❌ Error creating visualizations.")

print("✅ Main application logic defined!")


## 🧪 Testing and Examples

This section provides examples and testing functions for the weather application.

### What this section does:
- **Function Testing**: Demonstrates how to test individual functions
- **Example Usage**: Shows how to use the application programmatically
- **Sample Data**: Provides test cases for different scenarios
- **Debugging**: Helps identify issues during development

### Why it's modular:
Testing functions are separate from the main application logic, making it easy to run tests without affecting the main application. This follows the testing best practices and makes the code more reliable.


In [ ]:
# Test the weather data function
def test_weather_data():
    """Test the weather data retrieval function."""
    print("🧪 Testing weather data retrieval...")
    
    # Test with a known city
    test_location = "London"
    weather_data = get_weather_data(test_location, 3)
    
    if weather_data.get("success"):
        print(f"✅ Successfully retrieved weather data for {test_location}")
        print(f"   Current temperature: {weather_data['current']['temperature']['celsius']}°C")
        print(f"   Condition: {weather_data['current']['condition']}")
        print(f"   Forecast days: {len(weather_data['forecast'])}")
    else:
        print(f"❌ Failed to retrieve weather data: {weather_data.get('error')}")

# Test the natural language processing
def test_nlp_functions():
    """Test the natural language processing functions."""
    print("\n🧪 Testing NLP functions...")
    
    test_questions = [
        "What's the temperature in Paris?",
        "Will it rain tomorrow in Tokyo?",
        "How windy is it in New York?",
        "What's the weather like today?"
    ]
    
    for question in test_questions:
        print(f"\nQuestion: {question}")
        parsed = parse_weather_question(question)
        print(f"Parsed: {parsed}")

# Test visualization functions
def test_visualizations():
    """Test the visualization functions."""
    print("\n🧪 Testing visualization functions...")
    
    # Get sample weather data
    weather_data = get_weather_data("Sydney", 5)
    
    if weather_data.get("success"):
        print("Creating temperature visualization...")
        create_temperature_visualisation(weather_data)
        
        print("Creating precipitation visualization...")
        create_precipitation_visualisation(weather_data)
    else:
        print(f"❌ Cannot test visualizations: {weather_data.get('error')}")

# Run all tests
def run_all_tests():
    """Run all test functions."""
    print("🚀 Running all tests...")
    print("=" * 50)
    
    test_weather_data()
    test_nlp_functions()
    test_visualizations()
    
    print("\n✅ All tests completed!")

print("✅ Testing functions defined!")


## 🚀 Run the Application

This cell starts the Weather Advisor Application. Run this cell to begin using the application interactively.

### How to use:
1. **Run this cell** to start the application
2. **Follow the menu prompts** to navigate through different features
3. **Type '5'** to exit the application when you're done

### Features available:
- Get current weather for any location
- View multi-day weather forecasts
- Ask natural language weather questions
- Create weather data visualizations


In [ ]:
# Uncomment the line below to run the application
# run_weather_app()

# Uncomment the line below to run tests instead
# run_all_tests()

print("🌦️ Weather Advisor Application is ready!")
print("Uncomment one of the lines above to either:")
print("1. Run the interactive application (run_weather_app())")
print("2. Run the test suite (run_all_tests())")
